# DriftWatch - Data Ingestion & Preprocessing Exploration

This notebook explores and defines the preprocessing logic for the IBM Telco Customer Churn dataset before migrating it to `preprocess.py` for SageMaker Processing.

### Step 1: Load Data & Preview Head

In [6]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Set path relative to the project root / notebook directory
DATA_PATH = Path("../../data/raw/telco_churn.csv")

# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Display head
df.head()

Dataset shape: 7043 rows, 33 columns



,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [7]:
# Quick inspect of column types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

In [8]:
df.describe()

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,1.000000,100.000000,6500.000000


### Step 2: Data Quality & Column Inspection
- Check target distribution (`Churn Value`)
- Identify leakage & post-event columns
- Inspect blank string values in `Total Charges`
- Check for constant (single-value) columns

In [9]:
# 1. Target balance & leakage column verification
print("=== Target Distribution (Churn Value) ===")
print(df['Churn Value'].value_counts(normalize=True))
print(f"\nChurn Reason count: {df['Churn Reason'].notnull().sum()} (matches churners: {df['Churn Value'].sum()})")

# 2. Inspect blank whitespace entries in Total Charges
blank_total_charges = df[df['Total Charges'].str.strip() == '']
print(f"\nRecords with blank Total Charges: {len(blank_total_charges)}")
display(blank_total_charges[['Tenure Months', 'Monthly Charges', 'Total Charges']].head())

# 3. Check for constant columns (0 variance)
single_val_cols = [c for c in df.columns if df[c].nunique() == 1]
print(f"\nColumns with only 1 unique value: {single_val_cols}")

=== Target Distribution (Churn Value) ===
Churn Value
0    0.73463
1    0.26537
Name: proportion, dtype: float64

Churn Reason count: 1869 (matches churners: 1869)

Records with blank Total Charges: 11


,Tenure Months,Monthly Charges,Total Charges
2234,0,52.55,
2438,0,20.25,
2568,0,80.85,
2667,0,25.75,
2856,0,56.05,



Columns with only 1 unique value: ['Count', 'Country', 'State']


### Step 3: Drop Unneeded Columns (Leakage, Zero-Variance, High-Cardinality & Identifiers)

- **Identifiers**: `CustomerID` (100% unique IDs, cause overfitting)
- **Zero-Variance / Constants**: `Count`, `Country`, `State` (contain single constant values across all rows)
- **Data Leakage & Post-Event**: `Churn Label` (redundant string), `Churn Reason` (only known post-churn), `Churn Score`, `CLTV` (downstream business metrics)
- **High-Cardinality Geospatial**: `City`, `Zip Code`, `Lat Long`, `Latitude`, `Longitude` (avoid dimensionality explosion)

In [10]:
# Define columns to drop
COLUMNS_TO_DROP = [
    # Identifiers
    'CustomerID',
    # Zero-variance / constant
    'Count',
    'Country',
    'State',
    # Target leakage & post-event features
    'Churn Label',
    'Churn Reason',
    'Churn Score',
    'CLTV',
    # High-cardinality & geospatial
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude',
]

df_cleaned = df.drop(columns=COLUMNS_TO_DROP)
print(f"Original columns: {df.shape[1]} -> Remaining columns: {df_cleaned.shape[1]}")
print(f"Remaining columns ({len(df_cleaned.columns)}): {list(df_cleaned.columns)}")
df_cleaned.head()

Original columns: 33 -> Remaining columns: 20
Remaining columns (20): ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value']


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,1
3,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,1


### Step 4: Standardize Column Names & Clean Total Charges

- Standardize column names to snake_case / identifier-friendly names (`Tenure Months` -> `tenure`, `Monthly Charges` -> `MonthlyCharges`, `Total Charges` -> `TotalCharges`, `Churn Value` -> `Churn`).
- Convert `TotalCharges` from string to numeric, coercing spaces/blanks to `NaN`.
- Impute missing `TotalCharges` with `0.0` (as all 11 records have `tenure == 0`).

In [11]:
# Rename columns to standard names
column_mapping = {
    'Tenure Months': 'tenure',
    'Monthly Charges': 'MonthlyCharges',
    'Total Charges': 'TotalCharges',
    'Churn Value': 'Churn'
}
df_cleaned = df_cleaned.rename(columns=column_mapping)

# Convert TotalCharges to float and impute tenure=0 blanks with 0.0
df_cleaned['TotalCharges'] = pd.to_numeric(df_cleaned['TotalCharges'], errors='coerce')
df_cleaned['TotalCharges'] = df_cleaned['TotalCharges'].fillna(0.0)

print("Missing values after imputation:")
print(df_cleaned[['tenure', 'MonthlyCharges', 'TotalCharges']].isna().sum())
display(df_cleaned[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head())

Missing values after imputation:
tenure            0
MonthlyCharges    0
TotalCharges      0
dtype: int64


,tenure,MonthlyCharges,TotalCharges,Churn
0,2,53.85,108.15,1
1,2,70.70,151.65,1
2,8,99.65,820.50,1
3,28,104.80,3046.05,1
4,49,103.70,5036.30,1


### Step 5: One-Hot Encode Categorical Features

- Identify all categorical columns (`object` types).
- One-hot encode using `pd.get_dummies(..., drop_first=True, dtype=int)` to avoid collinearity (dummy variable trap).

In [12]:
# Identify categorical columns (excluding target)
categorical_cols = df_cleaned.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns to encode ({len(categorical_cols)}): {categorical_cols}")

# Perform one-hot encoding
df_encoded = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True, dtype=int)
print(f"Shape before encoding: {df_cleaned.shape} -> Shape after encoding: {df_encoded.shape}")
display(df_encoded.head())

Categorical columns to encode (16): ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']
Shape before encoding: (7043, 20) -> Shape after encoding: (7043, 31)


C:\Users\Dell\AppData\Local\Temp\ipykernel_17928\3913359869.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_cleaned.select_dtypes(include=['object', 'category']).columns.tolist()


,tenure,MonthlyCharges,TotalCharges,Churn,Gender_Male,Senior Citizen_Yes,Partner_Yes,Dependents_Yes,Phone Service_Yes,Multiple Lines_No phone service,...,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Paperless Billing_Yes,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,2,53.85,108.15,1,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
1,2,70.70,151.65,1,0,0,0,1,1,0,...,0,0,0,0,0,0,1,0,1,0
2,8,99.65,820.50,1,0,0,0,1,1,0,...,0,1,0,1,0,0,1,0,1,0
3,28,104.80,3046.05,1,0,0,1,1,1,0,...,0,1,0,1,0,0,1,0,1,0
4,49,103.70,5036.30,1,1,0,0,1,1,0,...,0,1,0,1,0,0,1,0,0,0


### Step 6: Stratified Train / Validation / Test Split (70% / 15% / 15%)

- Stratified on `Churn` to maintain the ~26.5% minority class ratio across all splits.
- Splitting BEFORE scaling prevents data leakage from validation/test sets into training.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# Split into 70% Train, 30% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Split Temp into 15% Validation, 15% Test (50/50 of temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train set: {len(X_train)} rows ({len(X_train)/len(df):.1%}) | Churn rate: {y_train.mean():.2%}")
print(f"Val set:   {len(X_val)} rows ({len(X_val)/len(df):.1%}) | Churn rate: {y_val.mean():.2%}")
print(f"Test set:  {len(X_test)} rows ({len(X_test)/len(df):.1%}) | Churn rate: {y_test.mean():.2%}")

Train set: 4930 rows (70.0%) | Churn rate: 26.53%
Val set:   1056 rows (15.0%) | Churn rate: 26.52%
Test set:  1057 rows (15.0%) | Churn rate: 26.58%


### Step 7: Feature Scaling with StandardScaler

- Fit `StandardScaler` **only** on `X_train` to prevent data leakage.
- Transform `X_train`, `X_val`, and `X_test`.

In [17]:
from sklearn.preprocessing import StandardScaler

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()

# Fit on train only, transform all splits
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Train numerical features summary after scaling:")
display(X_train[num_cols].describe())

Train numerical features summary after scaling:


,tenure,MonthlyCharges,TotalCharges
count,4.930000e+03,4.930000e+03,4.930000e+03
mean,1.167423e-16,4.143632e-17,-4.359821e-17
std,1.000101e+00,1.000101e+00,1.000101e+00
min,-1.317619e+00,-1.539324e+00,-1.004128e+00
25%,-9.524160e-01,-9.738588e-01,-8.312369e-01
50%,-1.408545e-01,1.860055e-01,-3.988937e-01
75%,9.547535e-01,8.300071e-01,6.784417e-01
max,1.604003e+00,1.779062e+00,2.784499e+00


### Step 8: Format for SageMaker XGBoost & Export Baseline

- **SageMaker XGBoost CSV Format**: Target `Churn` must be the **1st column**, with **no headers** and **no index**.
- **Model Monitor Baseline**: Export `baseline.csv` containing unlabeled features from `X_train` **with headers**.

In [25]:
# Combine target as 1st column for SageMaker XGBoost format
train_sagemaker = pd.concat([y_train.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
val_sagemaker = pd.concat([y_val.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
test_sagemaker = pd.concat([y_test.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)

# Create processed data directory
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save SageMaker XGBoost datasets (NO header, NO index)
train_sagemaker.to_csv(PROCESSED_DIR / "train.csv", index=False, header=False)
val_sagemaker.to_csv(PROCESSED_DIR / "validation.csv", index=False, header=False)
test_sagemaker.to_csv(PROCESSED_DIR / "test.csv", index=False, header=False)

# Save Model Monitor baseline (features only, WITH header, NO index)
X_train.to_csv(PROCESSED_DIR / "baseline.csv", index=False, header=True)

print(f"Successfully saved processed datasets to {PROCESSED_DIR}:")
print(f" - train.csv: {train_sagemaker.shape}")
print(f" - validation.csv: {val_sagemaker.shape}")
print(f" - test.csv: {test_sagemaker.shape}")
print(f" - baseline.csv: {X_train.shape}")

Successfully saved processed datasets to ..\..\data\processed:
 - train.csv: (4930, 31)
 - validation.csv: (1056, 31)
 - test.csv: (1057, 31)
 - baseline.csv: (4930, 30)
